# 05 - Evaluacion sobre Test

Carga ambos modelos entrenados y evalua sobre `splits/test/`. Genera matrices de confusion y reporta F1 por clase.


In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_score, recall_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

OUT = Path("./outputs")
SPLIT = Path("./splits")


class AsymmetricLoss(tf.keras.losses.Loss):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, label_smoothing=0.05,
                 class_weights=None, name="asymmetric_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights
        self.eps = 1e-8

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_s = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / num_classes
        xs_pos = y_pred
        xs_neg = 1.0 - y_pred
        if self.clip > 0:
            xs_neg = tf.clip_by_value(xs_neg + self.clip, 0.0, 1.0)
        log_pos = tf.math.log(tf.clip_by_value(xs_pos, self.eps, 1.0))
        log_neg = tf.math.log(tf.clip_by_value(xs_neg, self.eps, 1.0))
        loss_pos = y_true_s * log_pos
        loss_neg = (1.0 - y_true_s) * log_neg
        if self.gamma_pos > 0:
            loss_pos = loss_pos * tf.pow(1.0 - xs_pos, self.gamma_pos)
        if self.gamma_neg > 0:
            loss_neg = loss_neg * tf.pow(1.0 - xs_neg, self.gamma_neg)
        per_class = loss_pos + loss_neg
        if self.class_weights is not None:
            per_class = per_class * tf.constant(self.class_weights, tf.float32)
        return -tf.reduce_sum(per_class, axis=-1)

    def get_config(self):
        base = super().get_config()
        base.update({
            "gamma_neg": self.gamma_neg,
            "gamma_pos": self.gamma_pos,
            "clip": self.clip,
            "label_smoothing": self.label_smoothing,
            "class_weights": self.class_weights,
        })
        return base


_custom = {"AsymmetricLoss": AsymmetricLoss}

m1 = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2 = tf.keras.models.load_model(OUT / "model2_pathogen.keras", custom_objects=_custom)
print("Modelos cargados OK")

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from pathlib import Path
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input


class LeafSequence(Sequence):
    def __init__(self, directory, img_size=(224, 224), batch_size=32, class_mode="binary"):
        self.img_size = img_size
        self.batch_size = batch_size
        self.class_mode = class_mode
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))
        self.n = len(self.samples)
        self.classes = np.array([s[1] for s in self.samples])

    def __len__(self):
        return max(1, (self.n + self.batch_size - 1) // self.batch_size)

    def __getitem__(self, i):
        batch = self.samples[i * self.batch_size:(i + 1) * self.batch_size]
        imgs, labels = [], []
        for fp, label in batch:
            img = np.array(Image.open(fp).convert("RGB").resize(self.img_size))
            img = preprocess_input(img.astype(np.float32))
            imgs.append(img)
            labels.append(label)
        X = np.stack(imgs)
        if self.class_mode == "binary":
            Y = np.array(labels, dtype=np.float32)
        else:
            Y = np.eye(len(self.class_indices))[labels]
        return X, Y


In [ ]:
from sklearn.metrics import precision_recall_curve

test1 = LeafSequence(
    SPLIT / "test" / "clasificacion_binaria",
    img_size=(240, 240), batch_size=32, class_mode="binary",
)
val1 = LeafSequence(
    SPLIT / "clasificacion_binaria" / "val",
    img_size=(240, 240), batch_size=32, class_mode="binary",
)

_enf_idx = test1.class_indices.get("enferma", 0)
_sana_idx = test1.class_indices.get("sana", 1)

p_sana_test = m1.predict(test1, verbose=1).flatten()
p_enf_test = 1.0 - p_sana_test
reales1 = test1.classes[:len(p_enf_test)]
y_enf_test = (reales1 == _enf_idx).astype(int)

p_sana_val = m1.predict(val1, verbose=0).flatten()
p_enf_val = 1.0 - p_sana_val
y_enf_val = (val1.classes[:len(p_enf_val)] == _enf_idx).astype(int)

prec, rec, thr = precision_recall_curve(y_enf_val, p_enf_val)
_ok = rec[:-1] >= 0.90
gate = float(thr[_ok][int(np.argmax(prec[:-1][_ok]))]) if _ok.any() else 0.5
gate = round(float(np.clip(gate, 0.15, 0.5)), 3)
json.dump({"diseased_gate": gate}, open(OUT / "hs_threshold.json", "w"))
print(f"Umbral enferma tuneado (val, recall>=0.90): diseased_gate={gate}")


def _confusion_enf(p_enf, gate_val):
    pred_enf = (p_enf >= gate_val).astype(int)
    pred_lbl = np.where(pred_enf == 1, _enf_idx, _sana_idx)
    return confusion_matrix(reales1, pred_lbl, labels=[_enf_idx, _sana_idx])


_names = ["enferma", "sana"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, g, title in [(axes[0], 0.5, "thr 0.5"), (axes[1], gate, f"thr tuneado {gate}")]:
    cm = _confusion_enf(p_enf_test, g)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=_names, yticklabels=_names, ax=ax)
    ax.set_title(f"M1 - {title}"); ax.set_ylabel("real"); ax.set_xlabel("pred")
plt.tight_layout()
plt.savefig(OUT / "cm_m1.png", dpi=120)
plt.show()

preds1 = (p_enf_test >= gate).astype(int)
preds1 = np.where(preds1 == 1, _enf_idx, _sana_idx)
print("=== Modelo 1 (umbral tuneado) ===")
print(classification_report(reales1, preds1,
      target_names=[k for k, _ in sorted(test1.class_indices.items(), key=lambda kv: kv[1])],
      digits=4))
_rec_enf = recall_score(y_enf_test, (p_enf_test >= gate).astype(int), zero_division=0)
print(f"Recall enferma @ gate {gate}: {_rec_enf:.4f}  (objetivo >= 0.90)")

In [ ]:
test2 = LeafSequence(
    SPLIT / "test" / "clasificacion_patogeno",
    batch_size=32, class_mode="categorical",
)
probs2 = m2.predict(test2, verbose=1)

thresholds_path = OUT / "thresholds.json"
if thresholds_path.exists():
    thresholds = json.loads(thresholds_path.read_text())
else:
    thresholds = {name: 0.5 for name in test2.class_indices}

names2 = list(test2.class_indices.keys())
threshold_vector = np.array([thresholds.get(name, 0.5) for name in names2])
preds_multilabel = (probs2 >= threshold_vector).astype(int)

true_multilabel = np.eye(len(names2))[test2.classes[:len(probs2)]].astype(int)

from sklearn.metrics import average_precision_score, hamming_loss

per_class_ap = {}
per_class_f1 = {}
for i, name in enumerate(names2):
    per_class_ap[name] = float(average_precision_score(true_multilabel[:, i], probs2[:, i]))
    per_class_f1[name] = float(f1_score(true_multilabel[:, i], preds_multilabel[:, i], zero_division=0))

mAP = float(np.mean(list(per_class_ap.values())))
f1_macro = float(np.mean(list(per_class_f1.values())))
hl = float(hamming_loss(true_multilabel, preds_multilabel))

print("=== Modelo 2 multi-label ===")
print(f"mAP: {mAP:.4f}  |  F1 macro: {f1_macro:.4f}  |  Hamming loss: {hl:.4f}")
print("\nPer-class:")
for name in names2:
    print(f"  {name:20s} AP={per_class_ap[name]:.3f}  F1={per_class_f1[name]:.3f}  thr={thresholds.get(name, 0.5):.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(names2))
ax.bar(x - 0.2, [per_class_ap[n] for n in names2], width=0.4, label="AP")
ax.bar(x + 0.2, [per_class_f1[n] for n in names2], width=0.4, label="F1")
ax.set_xticks(x)
ax.set_xticklabels(names2, rotation=30)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Modelo 2 multi-label - AP vs F1 por clase")
plt.tight_layout()
plt.savefig(OUT / "m2_per_class_metrics.png", dpi=120)
plt.show()

In [ ]:
metrics = {
    "m1": {
        "accuracy": float(accuracy_score(reales1, preds1)),
        "f1": float(f1_score(reales1, preds1, zero_division=0)),
        "precision": float(precision_score(reales1, preds1, zero_division=0)),
        "recall": float(recall_score(reales1, preds1, zero_division=0)),
    },
    "m2_multilabel": {
        "mAP": mAP,
        "f1_macro": f1_macro,
        "hamming_loss": hl,
        "per_class_ap": per_class_ap,
        "per_class_f1": per_class_f1,
        "thresholds": thresholds,
    },
}
with open(OUT / "training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\n=== Verificación de objetivos ===")
print(f"M1 accuracy: {metrics['m1']['accuracy']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['accuracy'] >= 0.85 else 'FALLA'}")
print(f"M1 F1:       {metrics['m1']['f1']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['f1'] >= 0.85 else 'FALLA'}")
print(f"M2 mAP:      {mAP:.4f}  (objetivo >= 0.85) {'OK' if mAP >= 0.85 else 'FALLA'}")
print(f"M2 macro F1: {f1_macro:.4f}  (objetivo >= 0.80) {'OK' if f1_macro >= 0.80 else 'FALLA'}")
for cls, f1v in per_class_f1.items():
    print(f"  {cls:20s}: F1={f1v:.4f}  {'OK' if f1v >= 0.70 else 'REVISAR'}")

In [ ]:
import cv2
import numpy as np

_HSV_GREEN_LOW  = np.array([20, 25, 30])
_HSV_GREEN_HIGH = np.array([90, 255, 255])
_MORPH_KERNEL   = np.ones((5, 5), np.uint8)
_MORPH_LEAF     = np.ones((25, 25), np.uint8)


def gray_world_white_balance(img_rgb: np.ndarray) -> np.ndarray:
    result = img_rgb.astype(np.float32)
    avg = result.reshape(-1, 3).mean(axis=0)
    gray = avg.mean()
    scale = gray / (avg + 1e-6)
    result *= scale
    return np.clip(result, 0, 255).astype(np.uint8)


def clahe_luminance(img_rgb: np.ndarray, clip=2.0, grid=(8, 8)) -> np.ndarray:
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def chromatic_normalize(img_rgb: np.ndarray) -> np.ndarray:
    return clahe_luminance(gray_world_white_balance(img_rgb))


def _detect_leaf_region(hsv: np.ndarray) -> np.ndarray:
    _, s, v = cv2.split(hsv)
    _, mask_v = cv2.threshold(v, 28, 255, cv2.THRESH_BINARY)
    _, mask_s = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    leaf = cv2.bitwise_or(mask_v, mask_s)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_CLOSE, _MORPH_LEAF)
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    return leaf


def make_pseudo_mask(img_rgb: np.ndarray) -> np.ndarray:
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    leaf_region = _detect_leaf_region(hsv)
    healthy = cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH)
    healthy = cv2.morphologyEx(healthy, cv2.MORPH_OPEN, _MORPH_KERNEL)
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    mask[leaf_region > 0] = 2
    mask[healthy > 0] = 1
    return mask


def severity_from_mask(mask: np.ndarray) -> float:
    leaf_px = np.sum(mask > 0)
    dis_px  = np.sum(mask == 2)
    return float(dis_px / leaf_px * 100) if leaf_px > 0 else 0.0


print('chromatic_normalize, make_pseudo_mask y severity_from_mask definidos OK')

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

OUT  = globals().get('OUT', Path('./outputs'))
SPLIT = globals().get('SPLIT', Path('./splits'))

_SEG_MODEL_PATH = OUT / 'model_seg.keras'
if not _SEG_MODEL_PATH.exists():
    _SEG_MODEL_PATH = OUT / 'model_seg_best.keras'

if not _SEG_MODEL_PATH.exists():
    print('WARNING: model_seg.keras no encontrado. Entrenar primero el notebook 04.')
    mseg_eval = None
else:
    def dice_loss(y_true, y_pred, smooth=1e-6):
        num_classes = 3
        y_true_oh = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), -1), num_classes)
        total = tf.constant(0.0)
        for c in range(num_classes):
            p = y_pred[..., c]
            t = tf.cast(y_true_oh[..., c], tf.float32)
            inter = tf.reduce_sum(p * t)
            union = tf.reduce_sum(p) + tf.reduce_sum(t)
            total += 1.0 - (2 * inter + smooth) / (union + smooth)
        return total / tf.cast(num_classes, tf.float32)

    def seg_loss(y_true, y_pred):
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        return tf.reduce_mean(ce) + dice_loss(y_true, y_pred)

    class DiceCoef(tf.keras.metrics.Metric):
        def __init__(self, class_id=0, name=None, **kwargs):
            super().__init__(name=name or f'dice_c{class_id}', **kwargs)
            self.class_id = class_id
            self.inter = self.add_weight(name='inter', shape=(), initializer='zeros')
            self.union = self.add_weight(name='union', shape=(), initializer='zeros')

        def get_config(self):
            base = super().get_config()
            base['class_id'] = self.class_id
            return base

        def update_state(self, y_true, y_pred, sample_weight=None):
            pred_c = tf.cast(tf.argmax(y_pred, -1) == self.class_id, tf.float32)
            true_c = tf.cast(tf.squeeze(y_true, -1) == self.class_id, tf.float32)
            self.inter.assign_add(tf.reduce_sum(pred_c * true_c))
            self.union.assign_add(tf.reduce_sum(pred_c) + tf.reduce_sum(true_c))

        def result(self):
            return (2 * self.inter + 1e-6) / (self.union + 1e-6)

        def reset_state(self):
            self.inter.assign(0.0)
            self.union.assign(0.0)

    mseg_eval = tf.keras.models.load_model(
        _SEG_MODEL_PATH,
        custom_objects={'seg_loss': seg_loss, 'DiceCoef': DiceCoef, 'dice_loss': dice_loss},
    )
    print(f'M_seg cargado: {_SEG_MODEL_PATH}')


def _collect_seg_images(split_dir, max_images=200):
    paths = []
    for cls_dir in Path(split_dir).iterdir():
        if cls_dir.is_dir():
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                paths.extend(cls_dir.glob(ext))
    return paths[:max_images]


if mseg_eval is not None:
    val_paths = _collect_seg_images(SPLIT / 'clasificacion_patogeno' / 'val')
    dice_sana_scores, dice_enf_scores = [], []
    iou_sana_scores,  iou_enf_scores  = [], []
    sev_hsv, sev_seg = [], []

    for fp in val_paths:
        raw = np.array(Image.open(fp).convert('RGB').resize((256, 256)))
        img_rgb = chromatic_normalize(raw)
        true_mask = make_pseudo_mask(img_rgb)
        inp = (img_rgb.astype(np.float32) / 255.0)[np.newaxis]
        pred_prob = mseg_eval.predict(inp, verbose=0)[0]
        pred_mask = np.argmax(pred_prob, axis=-1)

        for c, dscore, iscore in [(1, dice_sana_scores, iou_sana_scores),
                                   (2, dice_enf_scores,  iou_enf_scores)]:
            inter = np.sum((pred_mask == c) & (true_mask == c))
            pred_c = np.sum(pred_mask == c)
            true_c = np.sum(true_mask == c)
            union  = pred_c + true_c - inter
            dscore.append((2 * inter + 1e-6) / (pred_c + true_c + 1e-6))
            iscore.append((inter + 1e-6) / (union + 1e-6))

        sev_hsv.append(severity_from_mask(true_mask))
        sev_seg.append(severity_from_mask(pred_mask))

    print(f'\n=== Evaluacion M_seg (n={len(val_paths)}) ===')
    print(f'Dice hoja_sana:     {np.mean(dice_sana_scores):.3f}')
    print(f'Dice hoja_enferma:  {np.mean(dice_enf_scores):.3f}  (objetivo >= 0.60)')
    print(f'IoU  hoja_sana:     {np.mean(iou_sana_scores):.3f}')
    print(f'IoU  hoja_enferma:  {np.mean(iou_enf_scores):.3f}')

In [ ]:
if mseg_eval is not None and len(sev_hsv) > 1:
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import pearsonr

    y_true = np.array(sev_hsv)
    y_pred = np.array(sev_seg)

    def ccc(a, b):
        mu_a, mu_b = np.mean(a), np.mean(b)
        var_a, var_b = np.var(a), np.var(b)
        cov = np.cov(a, b)[0, 1]
        return 2 * cov / (var_a + var_b + (mu_a - mu_b) ** 2 + 1e-9)

    def bland_altman(a, b):
        diff = b - a
        bias = np.mean(diff)
        return bias, bias + 1.96 * np.std(diff), bias - 1.96 * np.std(diff)

    r, _         = pearsonr(y_true, y_pred)
    mae          = np.mean(np.abs(y_pred - y_true))
    rmse         = np.sqrt(np.mean((y_pred - y_true) ** 2))
    ccc_val      = ccc(y_true, y_pred)
    bias, lo_u, lo_l = bland_altman(y_true, y_pred)
    pct_10       = np.mean(np.abs(y_pred - y_true) <= 10) * 100
    pct_20       = np.mean(np.abs(y_pred - y_true) <= 20) * 100

    print('\n=== Metricas de severidad continua (M_seg vs HSV ref) ===')
    print(f'Pearson r:  {r:.3f}  (objetivo >= 0.85)')
    print(f'CCC (Lin):  {ccc_val:.3f}')
    print(f'MAE (%):    {mae:.2f}  (objetivo <= 15)')
    print(f'RMSE (%):   {rmse:.2f}')
    print(f'Bland-Altman sesgo: {bias:.2f}%  [LoA: {lo_l:.2f}%, {lo_u:.2f}%]')
    print(f'Dentro de +-10%: {pct_10:.1f}%  |  Dentro de +-20%: {pct_20:.1f}%')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(y_true, y_pred, alpha=0.5, s=20)
    axes[0].plot([0, 100], [0, 100], 'r--', label='identidad')
    axes[0].set_xlabel('Severidad HSV ref (%)')
    axes[0].set_ylabel('Severidad M_seg (%)')
    axes[0].set_title(f'Predicha vs referencia  (r={r:.3f}, CCC={ccc_val:.3f})')
    axes[0].legend(); axes[0].grid(True)

    mean_vals = (y_true + y_pred) / 2
    diff_vals = y_pred - y_true
    axes[1].scatter(mean_vals, diff_vals, alpha=0.5, s=20)
    axes[1].axhline(bias,  color='r', linestyle='-',  label=f'sesgo={bias:.2f}%')
    axes[1].axhline(lo_u,  color='r', linestyle='--', label=f'+1.96SD={lo_u:.2f}%')
    axes[1].axhline(lo_l,  color='r', linestyle='--', label=f'-1.96SD={lo_l:.2f}%')
    axes[1].axhline(0, color='gray', linestyle=':')
    axes[1].set_xlabel('Media (ref + pred) / 2 (%)')
    axes[1].set_ylabel('Diferencia pred - ref (%)')
    axes[1].set_title('Bland-Altman')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    OUT = globals().get('OUT', __import__('pathlib').Path('./outputs'))
    plt.savefig(OUT / 'mseg_severity_analysis.png', dpi=120)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.hist(y_pred, bins=20, edgecolor='black', color='steelblue', alpha=0.8)
    plt.xlabel('Severidad predicha M_seg (%)')
    plt.ylabel('Frecuencia')
    plt.title('Distribucion de severidad foliar (val set)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / 'mseg_severity_histogram.png', dpi=120)
    plt.show()